# DICE ITC 02: Core Results

This notebook covers the main performance, calibration, observability-head comparison, and workload-holdout robustness. It assumes the result files already exist.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import sys
import tempfile
import types

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## 2. Main DICE Performance

This section reports the main digital-twin results for the paper:
- workload-conditioned scoring,
- sequential decisioning,
- mechanism-level diagnosis,
- tier-level contribution summaries.


In [ ]:
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
stressor_full = pd.read_csv(OUT_FULL / 'stressor_metrics_final_config.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
mechanism = pd.read_csv(OUT_FULL / 'mechanism_group_summary.csv')
tier_contrib = pd.read_csv(OUT_FULL / 'stressor_tier_contributions.csv')

row_final = overall_full[overall_full['config'] == 'tier0_tier1_tier2'].iloc[0]
seq_final = sequential[sequential['config'] == 'tier0_tier1_tier2'].iloc[0]
diag_final = diagnosis[diagnosis['config'] == 'tier0_tier1_tier2'].iloc[0]

main_summary = pd.DataFrame([
    {
        'Final head': 'Tier-0/1/2',
        'ROC-AUC (WC)': float(row_final['roc_auc_wc']),
        'AUC-PR (WC)': float(row_final['pr_auc_wc']),
        'Benign alert rate': float(seq_final['benign_run_alert_rate']),
        'Detection rate': float(seq_final['anomaly_detect_rate']),
        'Median TTD (s)': float(seq_final['median_time_to_detect_s']),
        'Top-1 diagnosis': float(diag_final['top1_acc']),
        'Top-2 diagnosis': float(diag_final['top2_acc']),
    }
])

display(Markdown('### Final-head summary'))
display(main_summary.round(4))

display(Markdown('### Main tables used in the paper'))
display(overall_full.round(4))
display(sequential.round(4))
display(diagnosis.round(4))

display(Markdown('### Final-head mechanism and tier summaries'))
display(mechanism.round(4))
display(tier_contrib.round(4))

for path in [
    OUT_FULL / 'figures' / 'fig_run_score_boxplot_wc.png',
    OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png',
    OUT_FULL / 'figures' / 'fig_detection_latency.png',
]:
    if path.exists():
        display(Image(filename=str(path)))


## 3. Reliability Without Per-Workload Tuning

This section addresses the paper's central reliability question: can residual-based digital-twin scores be thresholded with a target false-alarm budget without workload-specific manual tuning?

One distinction matters throughout this section. Split-conformal calibration controls the false-alarm rate at the **block level** under the calibrated benign regime. The notebook also reports **run-level** alert rates, which can be higher because a long run contains many blocks and the alert logic aggregates across them.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')
TARGET_ALPHA = 0.05

reliability_rows = []
for cfg, d in case_pred.groupby('config', sort=False):
    benign = d[d['label'] == 0].copy()
    anomaly = d[d['label'] == 1].copy()
    reliability_rows.append({
        'config': cfg,
        'target_alpha': TARGET_ALPHA,
        'benign_block_false_alarm_rate': benign['n_block_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_persist_false_alarm_rate': benign['n_persist_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_run_false_alarm_rate': benign['run_alert'].mean(),
        'anomaly_run_detection_rate': anomaly['run_alert'].mean(),
        'median_anomaly_time_to_detect_s': anomaly.loc[anomaly['run_alert'] == 1, 'time_to_detect_s'].median(),
    })

reliability = pd.DataFrame(reliability_rows)
reliability_by_workload = (
    case_pred[case_pred['label'] == 0]
    .groupby(['config', 'workload'], sort=False)
    .apply(
        lambda x: pd.Series({
            'target_alpha': TARGET_ALPHA,
            'benign_block_false_alarm_rate': x['n_block_alerts'].sum() / x['n_blocks'].sum(),
            'benign_persist_false_alarm_rate': x['n_persist_alerts'].sum() / x['n_blocks'].sum(),
            'benign_run_false_alarm_rate': x['run_alert'].mean(),
        }),
        include_groups=False,
    )
    .reset_index()
)

(OUT_PAPER / 'full').mkdir(parents=True, exist_ok=True)
reliability.to_csv(OUT_PAPER / 'full' / 'conformal_reliability_summary.csv', index=False)
reliability_by_workload.to_csv(OUT_APPENDIX / 'full' / 'conformal_reliability_by_workload.csv', index=False)

cfg_label_map = {'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'}
reliability['label'] = reliability['config'].map(cfg_label_map)

fig, ax = plt.subplots(figsize=(8.6, 4.6))
x = np.arange(len(reliability))
ax.plot(x, reliability['target_alpha'], color='black', linestyle='--', linewidth=2, label='Target alpha')
ax.scatter(x, reliability['benign_block_false_alarm_rate'], s=120, color='#E15759', label='Observed benign block FAR')
ax.scatter(x, reliability['benign_run_false_alarm_rate'], s=120, color='#4E79A7', label='Observed benign run FAR')
for i, row in reliability.iterrows():
    ax.text(i, row['benign_block_false_alarm_rate'] + 0.015, row['label'], ha='center', fontsize=10)
ax.set_xticks([])
ax.set_ylim(0, max(0.12, reliability[['target_alpha', 'benign_block_false_alarm_rate', 'benign_run_false_alarm_rate']].max().max() + 0.04))
ax.set_ylabel('False-alarm rate')
ax.set_title('Split-conformal reliability without per-workload tuning', fontweight='bold')
ax.legend(frameon=False)
ax.grid(alpha=0.20)
reliability_png = OUT_PAPER / 'full' / 'fig_conformal_reliability.png'
fig.tight_layout()
fig.savefig(reliability_png, dpi=220, bbox_inches='tight')
plt.close(fig)

display(reliability.round(4))
display(Image(filename=str(reliability_png)))

display(Markdown('### Benign false-alarm rate by workload'))
display(reliability_by_workload.round(4))


## 4. Observability Heads and Digital-Twin Variants

This section isolates what changes when DICE moves from `Tier-0` to `Tier-0/1` to `Tier-0/1/2`.
It separates the base digital-twin score, the workload-conditioned score, and the diagnosis head.


In [ ]:
diag = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
variant_rows = []
for _, row in overall_full.iterrows():
    cfg = row['config']
    drow = diag[diag['config'] == cfg].iloc[0]
    variant_rows.append({
        'config': cfg,
        'single_head_roc_auc': row['roc_auc'],
        'single_head_pr_auc': row['pr_auc'],
        'workload_conditioned_roc_auc': row['roc_auc_wc'],
        'workload_conditioned_pr_auc': row['pr_auc_wc'],
        'mechanism_top1_acc': drow['top1_acc'],
        'mechanism_top2_acc': drow['top2_acc'],
        'mechanism_macro_f1': drow['macro_f1'],
    })
variant_summary = pd.DataFrame(variant_rows)
variant_summary['label'] = variant_summary['config'].map({'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'})
variant_summary.to_csv(OUT_PAPER / 'full' / 'digital_twin_variant_summary.csv', index=False)
display(variant_summary.round(4))


## 5. Workload-Holdout Robustness and Drift Proxy

The draft frames workload/software drift as a practical in-field challenge.
This section uses workload holdout as the released robustness proxy.


In [ ]:
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    display(holdout.round(4))

    holdout_plot = holdout.copy()
    holdout_plot['label'] = holdout_plot['config'].map({'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'})
    fig, ax = plt.subplots(figsize=(8.4, 4.8))
    x = np.arange(len(holdout_plot))
    ax.bar(x - 0.15, holdout_plot['mean_pr_auc'], width=0.30, color='#4E79A7', label='Mean holdout AUC-PR')
    ax.bar(x + 0.15, holdout_plot['worst_pr_auc'], width=0.30, color='#E15759', label='Worst-workload AUC-PR')
    ax.set_xticks(x)
    ax.set_xticklabels(holdout_plot['label'])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('AUC-PR')
    ax.set_title('Workload-holdout robustness', fontweight='bold')
    ax.legend(frameon=False)
    ax.grid(axis='y', alpha=0.20)
    holdout_png = OUT_PAPER / 'full' / 'fig_holdout_robustness.png'
    fig.tight_layout()
    fig.savefig(holdout_png, dpi=220, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(holdout_png)))
else:
    display(Markdown('No holdout summary found yet.'))


## Next Step

Move to `dice_itc_03_dse_and_complexity.ipynb` for design-space exploration and the projected accelerator estimate.
